# 05 · RAG opción A: ejemplos etiquetados recuperados (dev)

> ## EJECUTAR EN KAGGLE
> - **Acelerador:** GPU T4 ×2
> - **Internet:** activado
> - **Secretos:** `HF_TOKEN` (y `GH_TOKEN` si el repo es privado)
> - **Ejecutar después de:** 03_llm.ipynb
> - **Entradas:** repo clonado; train como reserva de ejemplos, dev para evaluar; embedder `BAAI/bge-m3`
> - **Salidas (descargar y colocar en el repo):** `results/llm/ragA_dev_*.csv` → `results/llm/`; `results/rag/indice_bge-m3_train.npz` y `results/rag/vecinos_dev.csv` → `results/rag/`
> - **Si la sesión se corta:** volver a ejecutar; cada experimento reanuda desde su CSV (hay que conservar los CSV parciales, ver `doc/interno/KAGGLE.md`).


Para cada noticia de dev se recuperan las noticias de **train** más parecidas (embeddings multilingües, trozos de 300 palabras con 50 de solape) y se usan como ejemplos few-shot.

- La propia noticia se excluye siempre (`split:Id`); al evaluar dev la reserva es solo train.
- **8 vecinos equilibrados** (4 Fake + 4 Verdadera, los más parecidos de cada clase), con el mismo prompt que la configuración elegida en `03_llm` para cada modelo (`gemma4-31b`: `base`; `qwen38-27b`: `detallado`). Así la única diferencia con el few-shot fijo k=8 de 03 es **qué** ejemplos entran: vecinos frente a ejemplos fijos.
- **Referencia a batir:** el few-shot fijo k=8 de 03 (AUC dev 0,988 con `gemma4-31b`).
- No se muestran medios ni dominios: los ejemplos pasan por la misma limpieza que la noticia.

**Limitación.** Train y dev son México 2018 y no contienen **COVID-19**. En test (60 % COVID) los vecinos serán de otros temas: aportarán estilo y formato, no hechos. Lo que se mida aquí en dev puede no trasladarse; por eso se mira aparte el subgrupo de Fake no satíricas. **Test no se usa en este notebook.**

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/RAG"
PAQUETES = ['transformers', 'accelerate', 'bitsandbytes', 'sentence-transformers']

if IN_KAGGLE:
    os.environ["HF_HOME"] = "/tmp/hf"   # más disco libre que /kaggle/working
    ROOT = Path("/kaggle/working/Practica-TAPLN")

    def secreto(nombre):
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nombre)
        except Exception:
            return None

    if not ROOT.exists():
        gh = secreto("GH_TOKEN")   # solo hace falta si el repo es privado
        url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(ROOT)], check=True)
    if PAQUETES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PAQUETES], check=True)
    hf = secreto("HF_TOKEN")
    if hf:
        os.environ["HF_TOKEN"] = hf
    else:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos.")
    # Resultados intermedios no versionados: dataset privado `fakedes-resultados`
    extra = Path("/kaggle/input/fakedes-resultados/results")
    if extra.exists():
        shutil.copytree(extra, ROOT / "results", dirs_exist_ok=True)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

In [ ]:
import numpy as np
import pandas as pd
import torch
import data_utils as du
import preprocessing as pp
import evaluation as ev
import experiments as ex
import llm
import llm_fewshot as fs

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
HAY_GPU = torch.cuda.is_available()
MAX_TOKENS = 4500   # si un prompt no cabe, se quitan ejemplos; la noticia no se recorta 
print({k: len(v) for k, v in splits.items()}, "| GPU:", HAY_GPU)

## Índice de ejemplos (train)

In [ ]:
from rag.index import ExampleIndex, STEmbedder, TfidfEmbedder, EMBEDDER

RAG = ROOT / "results" / "rag"
RAG.mkdir(parents=True, exist_ok=True)
pool = ex.pool_for(splits, media_re, "dev")          # solo train
ruta_indice = RAG / "indice_bge-m3_train.npz"
try:
    emb = STEmbedder(EMBEDDER)                     
    ix = ExampleIndex.load(ruta_indice, emb) if ruta_indice.exists() else ExampleIndex(emb).build(pool)
    ix.save(ruta_indice)
except ImportError:
    print("Sin sentence-transformers: se usa TF-IDF como sustituto (solo para probar el notebook en local).")
    ix = ExampleIndex(TfidfEmbedder()).build(pool)
print(f"{len(ix.docs)} noticias, {len(ix.chunks)} trozos, embedder: {ix.embedder.name}")

### Calidad de la recuperación (descriptivo)

¿Los vecinos comparten etiqueta con la noticia? Se mide con un voto de los 5 vecinos de train (no es un sistema: solo indica cuánta señal lleva la recuperación).

In [ ]:
dev_in = ex.inputs_for(splits, media_re, "dev")
filas = []
for r in dev_in.itertuples(index=False):
    v = ix.search(r.body, 5, exclude=[f"dev:{r.Id}"], splits=("train",))
    filas.append({"Id": r.Id, "label": r.label, "frac_fake_vecinos": v["label"].mean(), "sim_max": v["sim"].max(),
                  "vecinos": "|".join(v["uid"])})
vec = pd.DataFrame(filas)
vec.to_csv(RAG / "vecinos_dev.csv", index=False)
print(ev.evaluate(vec.label, ev.predict_threshold(vec.frac_fake_vecinos, 0.5), vec.frac_fake_vecinos, "voto 5 vecinos"))
vec.sim_max.describe().round(3)

## Experimentos

Los vecinos de cada noticia se calculan al crear el experimento; después se libera el embedder para dejar la GPU al LLM.

In [ ]:
# Referencia a batir: la configuración elegida en 03 para cada modelo (few-shot fijo, k=8)
PROMPT = {"gemma4-31b": "base", "qwen38-27b": "detallado"}
REF = {"gemma4-31b": "ajuste_dev_gemma4-31b__base_k8.csv", "qwen38-27b": "ajuste_dev_qwen38-27b__detallado_k8.csv"}
# Mismo k y mismo prompt que en 03: lo único que cambia es qué ejemplos entran 
EXPS = {clave: [ex.experiment(splits, media_re, "dev", modo="rag_a", k=8, prompt=PROMPT[clave], index=ix,
                              per_class=True, max_tokens=MAX_TOKENS)]
        for clave in ex.FINALISTAS}
for clave, es in EXPS.items():
    for e in es:
        print(clave, e["experimento"], e["pid"])
print(ex.show_prompt(EXPS["gemma4-31b"][0]))
if hasattr(ix.embedder, "model"):
    ix.embedder.model = None      
llm.release()


## Ejecución (GPU)

In [ ]:
MODELOS = list(ex.FINALISTAS)   # gemma4-31b (principal) y qwen38-27b (contraste)
if HAY_GPU:
    for clave in MODELOS:
        ex.run_all(ROOT, EXPS[clave], [clave], borrar_cache=IN_KAGGLE)
else:
    print("Sin GPU: no se ejecutan modelos. Con los CSV ya descargados en results/llm/ se puede ver el resumen.")

## Resultados en dev

In [ ]:
COLS = ["modelo", "decision", "f1_fake", "f1_fake_inf", "f1_fake_sup", "auc", "auc_inf", "auc_sup",
        "precision", "recall", "pct_pred_fake", "s_noticia", "tokens", "completo"]
tabla = ex.dev_table(ROOT, "dev")
if len(tabla):
    tabla = tabla.sort_values("auc", ascending=False)
    tabla.to_csv(ROOT / "results" / "llm" / "ragA_dev_resumen.csv", index=False)
    display(tabla[COLS].round(3).reset_index(drop=True))

In [ ]:
# Comparación pareada de cada experimento nuevo frente a la configuración elegida en 03
# (few-shot fijo k=8) y frente al zero-shot del piloto, por modelo
filas = []
for clave in MODELOS:
    sistemas = {}
    for nombre, f in [("few-shot k=8 (03)", REF[clave]), ("zero-shot (piloto)", f"piloto_dev_{clave}.csv")]:
        if (ROOT / "results" / "llm" / f).exists():
            sistemas[nombre] = ex.to_predictions(pd.read_csv(ROOT / "results" / "llm" / f), nombre)
    for e in EXPS[clave]:
        p = ex.out_path(ROOT, e, clave)
        if fs.is_complete(p, len(e["inputs"])):
            sistemas[f"{e['experimento']} {e['pid']}"] = ex.to_predictions(pd.read_csv(p), e["pid"])
    nuevos = [s for s in sistemas if s not in ("few-shot k=8 (03)", "zero-shot (piloto)")]
    if nuevos and len(sistemas) > 1:
        c = ev.compare_predictions(ROOT, sistemas, "dev")
        filas.append(c[c["A"].isin(nuevos) ^ c["B"].isin(nuevos)].assign(modelo=clave))
if filas:
    display(pd.concat(filas, ignore_index=True).round(3))


In [ ]:
# Desglose del mejor candidato de cada modelo: Fake satíricas / otras (dev) y COVID / no COVID
for clave in MODELOS:
    for e in EXPS[clave]:
        p = ex.out_path(ROOT, e, clave)
        if fs.is_complete(p, len(e["inputs"])):
            r = ev.evaluate_predictions(ROOT, ex.to_predictions(pd.read_csv(p), p.stem), "dev", p.stem, n_boot=200)
            d = r["desglose"]
            print(p.stem)
            display(d[d.decision == "umbral"][["desglose", "grupo", "n", "f1_fake", "auc", "pct_pred_fake"]].round(3))

## Lectura de resultados y decisión

**[PENDIENTE: rellenar después de ejecutar.]** Anotar aquí y en `doc/diario_trabajo.md`:

- Si mejora al few-shot fijo k=8 de `03_llm` (y al zero-shot) y si la diferencia es significativa (tabla pareada).
- AUC en las Fake no satíricas (es lo más parecido a test que hay en dev).
- Coste (tokens y s/noticia) y problemas encontrados.
- Si la aproximación se mantiene o se abandona, y por qué. **Test no se usa aquí.**
